# 05. Post-training: SFT, 선호 학습(DPO), 강화학습(GRPO)

ChatGPT, Claude, Qwen-Instruct 같은 "대화하는 모델"은 프리트레이닝만으로 만들어지지 않습니다.
프리트레이닝이 끝난 뒤에 하는 모든 학습을 묶어서 **post-training(사후 학습)** 이라고 부릅니다.
이번 단계에서는 post-training의 전체 그림을 보고, 그 안의 세 가지 핵심 기법을 직접 돌려 봅니다.

| 기법 | 한 줄 설명 | 이 과정에서 |
|---|---|---|
| **SFT** (Supervised Fine-Tuning) | 모범 답안을 그대로 따라 쓰게 학습 | 2·3단계에서 이미 했음 → 이번에 TRL로 다시 |
| **선호 학습** (RLHF, DPO) | 두 답 중 사람이 더 좋다고 고른 쪽을 더 잘 내도록 학습 | DPO 실습 |
| **강화학습** (PPO, GRPO) | 모델이 직접 답을 만들고, 점수(보상)를 받아 더 좋은 답 쪽으로 학습 | 장난감 예제 + GRPO 맛보기 |

**목표:** 2단계의 "고양이 비서 냥이"를 이어서,

1. **SFT** 로 냥이 말투를 가르치고 (2단계 복습, 이번엔 TRL `SFTTrainer` + LoRA)
2. **DPO** 로 "짧고 친절한 답"을 "길거나 무례한 답"보다 선호하게 만들고
3. **GRPO** 로 정답 없이 **채점 함수(보상)만으로** 말투를 강화해 봅니다.

**이 노트북에서 배우는 것**

0. LLM이 만들어지는 전체 과정: pretraining → SFT → 선호 학습 → 강화학습
1. SFT와 강화학습은 수식으로 보면 무엇이 같고 무엇이 다른가
2. TRL 라이브러리로 SFT 하기 (2단계에서 직접 짠 코드와 비교)
3. RLHF(PPO)의 구조와, 왜 무거운가
4. DPO의 원리를 직접 계산해 보고, `DPOTrainer`로 학습하기
5. 순수 PyTorch 장난감 예제로 강화학습(GRPO)의 핵심 이해하기
6. `GRPOTrainer`로 진짜 모델에 보상 기반 학습 맛보기, 그리고 reward hacking

**실행 환경**

| 환경 | 예상 시간 (전체) | 필요 메모리 |
|---|---|---|
| RTX 3060 12GB | 약 10~15분 | VRAM 약 4~6GB (LoRA라서 여유 있음) |
| CPU만 있는 경우 | 약 30~60분 (스텝 수를 줄인 설정) | RAM 8GB 이상 권장 |

> 💡 이 노트북은 3단계의 **LoRA**를 씁니다. LoRA는 "원래 가중치는 얼려 두고 작은 보조 행렬만 학습"하는 방법이라 메모리가 적게 듭니다.
> 3단계를 아직 안 봤다면 "가볍게 학습하는 방법" 정도로 이해하고 넘어가도 괜찮습니다.

> ⚠️ 처음 실행할 때 Hugging Face Hub에서 모델(약 1GB)을 내려받으므로 인터넷 연결이 필요합니다. 셀은 위에서부터 순서대로 실행하세요.

## 0. 큰 그림: LLM은 어떤 순서로 만들어지나

```
 ┌────────────────┐    ┌────────────────┐    ┌────────────────────┐    ┌─────────────────────┐
 │ ① Pretraining   │ →  │ ② SFT           │ →  │ ③ 선호 학습         │ →  │ ④ 강화학습 (RL)       │
 │ 인터넷 텍스트   │    │ (질문, 모범답안) │    │ (질문, 좋은 답,     │    │ 질문 + 채점 함수      │
 │ 다음 토큰 예측  │    │ 따라 쓰기        │    │  나쁜 답) 비교       │    │ 직접 풀고 점수 받기    │
 └────────────────┘    └────────────────┘    └────────────────────┘    └─────────────────────┘
   base model            instruct model         더 도움되고 안전한 답       추론(reasoning) 능력
   (1단계)               (2·3단계, 이번 3절)     (이번 4절: DPO)             (이번 5·6절: GRPO)
   ◀──────────── pretraining ──────────▶◀──────────────────── post-training ─────────────────────▶
```

| 단계 | 데이터 | 무엇을 배우나 | 실제 예 |
|---|---|---|---|
| ① Pretraining | 수조 개 토큰의 일반 텍스트 | 언어, 지식, 추론의 기초 | `Qwen/Qwen2.5-0.5B` (base) |
| ② SFT | 수천~수십만 개의 (지시, 모범 답변) | 질문에 답하는 **형식과 말투** | `Qwen/Qwen2.5-0.5B-Instruct`의 첫 단계 |
| ③ 선호 학습 | (질문, 더 좋은 답, 더 나쁜 답) 쌍 | 여러 괜찮은 답 중 **사람이 더 좋아하는 쪽** | ChatGPT의 RLHF, Llama/Qwen의 DPO |
| ④ 강화학습 | 질문 + **자동 채점기** (수학 정답 확인, 코드 테스트 등) | 스스로 시도하며 **정답을 맞히는 전략** | DeepSeek-R1, OpenAI o1 계열의 추론 모델 |

**비유하자면** 학생 한 명을 키우는 과정과 비슷합니다.

- ① Pretraining = 도서관의 책을 엄청나게 많이 읽기
- ② SFT = 선생님이 써 준 **모범 답안을 그대로 따라 쓰기**
- ③ 선호 학습 = 내가 쓴 답안 두 개를 보여 주면 선생님이 "**이쪽이 더 낫다**"고 골라 주기
- ④ 강화학습 = 문제집을 풀고 **채점만 받으면서** 스스로 요령을 찾기 (정답 풀이는 안 보여 줌)

> 📝 Hugging Face에서 모델 이름 끝에 `-Instruct`, `-Chat`이 붙어 있으면 ②~③까지 끝난 모델이고, 아무것도 안 붙어 있으면(`Qwen2.5-0.5B`) ①만 끝난 **base 모델**입니다.
> 2단계에서 base 모델이 질문에 엉뚱하게 답하던 것을 기억하나요? 그 모델을 "대화하는 모델"로 바꾸는 것이 post-training입니다.

## 1. SFT와 강화학습은 수식으로 보면 사촌이다

### SFT (지도 학습): 정답을 보여 주고 따라 하게 한다

2단계에서 쓴 loss를 다시 써 보면, 질문 $x$와 **사람이 써 준** 답 $y^*$에 대해

$$\mathcal{L}_{\text{SFT}} = -\log \pi_\theta(y^* \mid x) = -\sum_t \log \pi_\theta(y^*_t \mid x, y^*_{<t})$$

여기서 $\pi_\theta$는 모델(정책, policy)이고, 답변 토큰들의 log 확률 합에 마이너스를 붙인 것입니다. "정답 답변의 확률을 높여라"는 뜻입니다.

### 강화학습: 모델이 직접 답을 만들고, 점수만 받는다

강화학습에는 정답 $y^*$가 없습니다. 대신

1. 모델이 **직접 답 $y$를 생성**하고 (샘플링)
2. 채점기가 그 답에 **보상(reward) $r$** 을 줍니다 (좋으면 +, 나쁘면 −)
3. 보상에 비례해서 그 답의 확률을 높이거나 낮춥니다

$$\mathcal{L}_{\text{RL}} = -\, A \cdot \log \pi_\theta(y \mid x), \qquad y \sim \pi_\theta(\cdot \mid x)$$

$A$는 advantage라고 부르며, "이 답이 **평소보다** 얼마나 좋았나"(보상 − 평균 보상)입니다.

### 비교해 보면

| | SFT | 강화학습 |
|---|---|---|
| 답은 누가 만드나 | 사람 (데이터에 있음) | **모델 자신** (학습 중에 생성) |
| 신호 | "이게 정답이야" | "네 답은 0.8점이야" |
| loss의 가중치 | 항상 +1 (무조건 확률 ↑) | $A$: 좋은 답은 확률 ↑, **나쁜 답은 확률 ↓** |
| 필요한 것 | 모범 답안 | 채점 방법 (사람의 선호, 보상 모델, 규칙) |
| 장점 | 간단하고 안정적 | 모범 답안을 쓰기 어려워도 **채점만 할 수 있으면** 학습 가능. "하지 말아야 할 것"도 배움 |
| 단점 | 데이터 이상으로 잘하기 어려움 | 불안정, 느림(생성 필요), 채점기의 허점을 파고듦(reward hacking) |

즉, **SFT는 "보상이 항상 1이고 답을 사람이 준" 특수한 강화학습**이라고 볼 수 있습니다.
이 차이를 기억하면서 SFT → DPO → GRPO 순서로 실습해 봅시다.

## 2. 환경 확인과 실행 설정

이번에는 Hugging Face의 **TRL**(Transformer Reinforcement Learning) 라이브러리를 씁니다. SFT, DPO, GRPO, PPO 등 post-training 기법을 `Trainer`처럼 쓸 수 있게 만든 라이브러리입니다.
설치가 안 되어 있다면 `pip install -r requirements.txt` (또는 `pip install "trl>=1.0"`)를 실행하세요.

In [ ]:
import gc
import math
import os
import random
import time
from pathlib import Path

import torch
import transformers
import datasets
import peft
import trl

print("PyTorch      :", torch.__version__)
print("transformers :", transformers.__version__)
print("datasets     :", datasets.__version__)
print("peft         :", peft.__version__)
print("trl          :", trl.__version__)

if torch.cuda.is_available():
    device = "cuda"                      # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = "mps"                       # Apple Silicon Mac (M1~M4) GPU
else:
    device = "cpu"
print("사용할 장치  :", device)
if device == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU 이름     :", props.name)
    print(f"VRAM         : {props.total_memory / 1024**3:.1f} GB")

transformers.set_seed(42)

아래 `config`가 세 단계(SFT, DPO, GRPO)의 학습 설정입니다. CPU에서는 반복 횟수를 줄여서 흐름만 볼 수 있게 했습니다.

- **학습률:** LoRA는 학습하는 파라미터가 적어서 Full 파인튜닝(2단계, `3e-5`)보다 큰 학습률을 씁니다. DPO와 GRPO는 SFT 모델을 **조금만** 고쳐야 해서 SFT보다 작게 잡습니다.
- **`dpo_beta`, `grpo_beta`:** 원래 모델(기준 모델)에서 얼마나 멀어지는 것을 허용할지 정하는 값입니다. 4절, 6절에서 설명합니다.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"   # 2단계와 같은 base 모델에서 출발

if device in ("cuda", "mps"):
    # Mac(mps)은 GPU 설정을 쓰되, 혼합 정밀도(bf16/fp16) 학습은 MPS에서 불안정할 수 있어 fp32로 학습한다
    use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
    config = dict(
        bf16=use_bf16, fp16=device == "cuda" and not use_bf16,
        # SFT
        sft_epochs=3, sft_lr=2e-4, sft_batch_size=8,
        # DPO
        dpo_epochs=3, dpo_lr=5e-5, dpo_batch_size=4, dpo_grad_accum=2, dpo_beta=0.1,
        # GRPO
        grpo_steps=30, grpo_lr=2e-5, grpo_num_generations=4, grpo_batch_size=8, grpo_beta=0.04,
    )
else:
    config = dict(
        bf16=False, fp16=False,
        sft_epochs=2, sft_lr=3e-4, sft_batch_size=4,
        dpo_epochs=2, dpo_lr=8e-5, dpo_batch_size=2, dpo_grad_accum=2, dpo_beta=0.1,
        grpo_steps=6, grpo_lr=3e-5, grpo_num_generations=4, grpo_batch_size=4, grpo_beta=0.04,
    )
    torch.set_num_threads(os.cpu_count())

for k, v in config.items():
    print(f"{k:22}: {v}")

# 노트북은 notebooks/ 폴더에서 실행되므로 한 단계 위의 data/, outputs/ 폴더를 쓴다
ROOT = Path("..") if Path("../data").exists() else Path(".")
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs" / "05_post_training"

한글 그래프를 위한 폰트 설정과, 메모리를 비우는 작은 도우미 함수입니다.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

for font in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if any(f.name == font for f in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = font
        plt.rcParams["axes.unicode_minus"] = False
        break

def free_memory():
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    elif device == "mps":
        torch.mps.empty_cache()

def peak_vram():
    return torch.cuda.max_memory_allocated() / 1024**3 if device == "cuda" else float("nan")

### 토크나이저와 모델 불러오기, 프롬프트 형식

프롬프트 형식은 2단계와 똑같이 씁니다. (학습과 사용 시 같은 형식을 써야 한다는 것, 기억하죠?)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_prompt(question):
    return f"### 질문:\n{question}\n\n### 답변:\n"

@torch.no_grad()
def ask(model, question, max_new_tokens=60):
    model.eval()
    inputs = tokenizer(build_prompt(question), return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)   # fp32로 불러옴
print(f"파라미터 수: {sum(p.numel() for p in base_model.parameters()) / 1e6:.0f}M")

### 답변 품질을 숫자로 재는 함수

학습 전후를 눈으로만 비교하면 헷갈리기 쉬우니, 냥이 답변의 특징을 간단한 숫자로 재 봅니다.

- **평균 길이:** 짧고 간결할수록 좋다고 정할 것입니다
- **🐾로 끝남:** 냥이 답변의 마무리 규칙
- **'냥' 포함:** 냥이 말투
- **무례함:** "몰라?", "알아서" 같은 표현이 들어 있는지

In [ ]:
test_questions = [
    "하늘은 왜 파래?",               # 평가용 데이터 (학습에 안 씀)
    "GPU가 뭐야?",                   # 평가용 데이터
    "고양이는 몇 시간 자?",          # 평가용 데이터
    "서울에서 부산까지 어떻게 가?",  # 처음 보는 질문
    "LLM이 뭐야?",                   # 처음 보는 질문
    "오늘 저녁 메뉴 추천해 줘.",     # 처음 보는 질문
]

RUDE_WORDS = ["몰라?", "알아서", "그래서 뭐"]

def style_report(answers):
    n = len(answers)
    return {
        "평균 길이(글자)": sum(len(a) for a in answers) / n,
        "🐾로 끝남": sum(a.endswith("🐾") for a in answers) / n,
        "'냥' 포함": sum("냥" in a for a in answers) / n,
        "무례함": sum(any(w in a for w in RUDE_WORDS) for a in answers) / n,
    }

def evaluate_model(model, name):
    answers = [ask(model, q) for q in test_questions]
    report = style_report(answers)
    print(f"[{name}] " + ", ".join(f"{k}: {v:.2f}" for k, v in report.items()))
    return answers, report

all_answers, all_reports = {}, {}
all_answers["base"], all_reports["base"] = evaluate_model(base_model, "base")
for q, a in zip(test_questions[:3], all_answers["base"]):
    print(f"\nQ: {q}\nA: {a}")

base 모델은 형식도 말투도 모르고, 끝맺지 못하는 경우가 많습니다. 이제 post-training을 단계별로 해 봅시다.

## 3. Stage 1: SFT (지도 파인튜닝) with TRL

2단계에서는 SFT에 필요한 것들을 직접 만들었습니다.

1. 프롬프트 + 답변 + EOS를 이어 붙여 토큰화
2. 프롬프트 부분 labels를 `-100`으로 (답변만 loss)
3. 패딩하는 collate 함수
4. `Trainer`로 학습

TRL의 `SFTTrainer`는 데이터를 **`prompt` / `completion` 두 칸**으로 주면 1~3을 알아서 해 줍니다.
(`completion_only_loss=True`가 바로 2단계의 "프롬프트 `-100` 마스킹"입니다.)

In [ ]:
from datasets import load_dataset

sft_raw = load_dataset(
    "json",
    data_files={"train": str(DATA_DIR / "cat_assistant_train.jsonl"),
                "eval": str(DATA_DIR / "cat_assistant_eval.jsonl")},
)

# TRL이 이해하는 prompt / completion 형식으로 바꾸기
def to_prompt_completion(ex):
    return {"prompt": build_prompt(ex["question"]), "completion": ex["answer"]}

sft_data = sft_raw.map(to_prompt_completion, remove_columns=sft_raw["train"].column_names)
print(sft_data)
print("\nprompt     :", repr(sft_data["train"][0]["prompt"]))
print("completion :", repr(sft_data["train"][0]["completion"]))

LoRA 설정입니다. 3단계와 같은 방식으로, 어텐션과 MLP의 선형층에 작은 보조 행렬(rank `r=16`)을 붙입니다.

In [ ]:
from peft import LoraConfig

def make_lora_config():
    return LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "sft"),
    num_train_epochs=config["sft_epochs"],
    per_device_train_batch_size=config["sft_batch_size"],
    per_device_eval_batch_size=config["sft_batch_size"],
    learning_rate=config["sft_lr"],
    lr_scheduler_type="cosine",
    warmup_steps=2,
    max_length=256,
    completion_only_loss=True,     # 답변(completion) 부분만 loss → 2단계의 -100 마스킹
    bf16=config["bf16"],
    fp16=config["fp16"],
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)

sft_trainer = SFTTrainer(
    model=base_model,
    args=sft_args,
    train_dataset=sft_data["train"],
    eval_dataset=sft_data["eval"],
    processing_class=tokenizer,
    peft_config=make_lora_config(),
)
sft_trainer.model.print_trainable_parameters()

TRL이 실제로 2단계와 같은 데이터를 만들었는지 확인해 봅시다. labels 중 `-100`이 아닌 부분(= loss를 계산하는 부분)이 답변 + EOS여야 합니다.

In [ ]:
batch = sft_trainer.data_collator([sft_trainer.train_dataset[0]])
ids, labels = batch["input_ids"][0], batch["labels"][0]
print("전체 토큰      :", repr(tokenizer.decode(ids)))
print("loss 계산 부분 :", repr(tokenizer.decode(ids[labels != -100])))

In [ ]:
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
start = time.time()
sft_trainer.train()
print(f"\nSFT 학습 시간: {(time.time() - start) / 60:.1f}분, 최대 VRAM: {peak_vram():.2f} GB")

### LoRA 합치기 (merge)

LoRA 보조 행렬을 원래 가중치에 **더해서 하나의 모델로 합칩니다** ($W' = W + BA$).
합친 모델이 다음 단계(DPO)의 **출발점이자 기준(reference) 모델**이 됩니다. 실제 post-training도 이렇게 "SFT 모델을 만들고 → 그 위에서 선호 학습"하는 순서로 진행됩니다.

In [ ]:
sft_model = sft_trainer.model.merge_and_unload()
del sft_trainer, base_model
free_memory()

all_answers["sft"], all_reports["sft"] = evaluate_model(sft_model, "SFT")
for q, a in zip(test_questions, all_answers["sft"]):
    print(f"\nQ: {q}\nA: {a}")

2단계와 마찬가지로 냥이 말투로 답하고 🐾에서 멈출 것입니다. 하지만 SFT에는 한계가 있습니다.

- SFT는 **모범 답안만** 보여 줍니다. "이런 답은 하지 마"를 직접 가르칠 방법이 없습니다.
- 학습 데이터 답변이 대부분 두 문장이라서, 모델도 두 문장 정도로 답합니다. 우리가 "**한 문장으로 짧게**"를 더 좋아한다면? 모든 데이터를 다시 써야 합니다.
- 현실에서는 좋은 답을 **직접 쓰는 것보다, 두 답 중 더 나은 것을 고르는 것**이 훨씬 쉽고 저렴합니다.

여기서 선호 학습이 등장합니다.

> 💡 2단계를 실행해서 `outputs/02_full_ft/final`이 있다면, `sft_model = AutoModelForCausalLM.from_pretrained("../outputs/02_full_ft/final").to(device)`로 2단계의 Full 파인튜닝 모델을 SFT 모델로 써도 됩니다.

## 4. Stage 2: 선호 학습 (RLHF와 DPO)

### 4-1. 선호 데이터

선호 데이터는 **(질문, 더 좋은 답 `chosen`, 더 나쁜 답 `rejected`)** 의 묶음입니다.
`data/cat_preference_*.jsonl`에 이번 실습용 데이터를 준비했습니다. 기준은 "**한 문장으로 짧고 친절한 냥이 답변**"이 좋다는 것이고, 나쁜 답은 세 종류입니다.

| `rejected_type` | 나쁜 답의 종류 |
|---|---|
| `long` | SFT 데이터 그대로의 두 문장 답변 (나쁘진 않지만 덜 선호) |
| `rude` | 냥이 말투를 버리고 무례하게 답함 |
| `ramble` | 같은 말을 반복하며 장황하게 늘어놓고 끝맺지 못함 |

실제로는 같은 질문에 모델이 만든 답 여러 개를 사람(또는 더 큰 AI)이 비교해서 고르는 식으로 이런 데이터를 모읍니다.

In [ ]:
pref_raw = load_dataset(
    "json",
    data_files={"train": str(DATA_DIR / "cat_preference_train.jsonl"),
                "eval": str(DATA_DIR / "cat_preference_eval.jsonl")},
)
print(pref_raw)
for ex in pref_raw["train"].select(range(3)):
    print(f"\n질문    : {ex['prompt']}")
    print(f"chosen  : {ex['chosen']}")
    print(f"rejected: {ex['rejected']}  ({ex['rejected_type']})")

### 4-2. 원조 방법: RLHF (보상 모델 + PPO)

ChatGPT(InstructGPT)가 쓴 원래 방법은 두 단계입니다.

1. **보상 모델(reward model) 학습:** 선호 데이터로 "답을 보고 점수를 매기는 모델" $r_\phi(x, y)$를 학습합니다. chosen 점수 > rejected 점수가 되도록.
2. **PPO로 강화학습:** 정책 모델이 답을 생성 → 보상 모델이 채점 → 점수가 높아지는 쪽으로 학습. 단, 원래 SFT 모델에서 너무 멀어지지 않도록 **KL 페널티**를 겁니다.

$$\max_\theta \; \mathbb{E}_{y \sim \pi_\theta}\big[\, r_\phi(x, y) \,\big] \;-\; \beta \, \mathrm{KL}\big(\pi_\theta \,\|\, \pi_{\text{ref}}\big)$$

**KL 페널티가 왜 필요할까?** 보상 모델도 완벽하지 않습니다. 제약이 없으면 정책 모델은 보상 모델의 허점(예: 특정 단어만 반복하면 점수가 높음)을 찾아내서 이상한 답만 하게 됩니다. 이를 **reward hacking**이라고 합니다. 기준 모델 $\pi_{\text{ref}}$(SFT 모델)에서 크게 벗어나지 않게 붙잡아 두는 역할입니다.

**PPO의 문제: 너무 무겁다.** 학습 중에 메모리에 모델이 4개 필요합니다.

| 모델 | 역할 | 학습하나? |
|---|---|---|
| 정책 모델 $\pi_\theta$ | 답을 생성하고 학습됨 | ✅ |
| 기준 모델 $\pi_{\text{ref}}$ | KL 계산용 (SFT 모델 복사본) | ❌ |
| 보상 모델 $r_\phi$ | 점수 매기기 | ❌ |
| 가치 모델 (value/critic) | "이 상황에서 기대 점수"를 예측해서 advantage 계산 | ✅ |

게다가 매 스텝 답을 **생성**해야 해서 느리고, 하이퍼파라미터에 민감해서 불안정합니다. 그래서 등장한 것이 DPO입니다.

### 4-3. DPO: 보상 모델 없이, 선호 데이터로 바로 학습

DPO(Direct Preference Optimization, 2023)는 수학적으로 위 RLHF 목표의 최적해를 정리해서, **보상 모델과 PPO 없이 선호 데이터만으로 같은 목표를 학습**하게 만든 방법입니다.

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\Big(\beta \big[\underbrace{(\log \pi_\theta(y_w|x) - \log \pi_{\text{ref}}(y_w|x))}_{\text{chosen이 기준 대비 얼마나 올랐나}} - \underbrace{(\log \pi_\theta(y_l|x) - \log \pi_{\text{ref}}(y_l|x))}_{\text{rejected가 기준 대비 얼마나 올랐나}}\big]\Big)$$

- $y_w$ = chosen(win), $y_l$ = rejected(lose), $\sigma$ = 시그모이드
- 쉽게 말해 "**chosen의 확률은 기준 모델보다 올리고, rejected의 확률은 기준 모델보다 내려라**"
- $\beta \cdot (\log \pi_\theta - \log \pi_{\text{ref}})$를 **암묵적 보상(implicit reward)** 이라고 부릅니다. 모델 자신이 보상 모델 역할을 겸하는 셈입니다.
- 필요한 모델은 정책 모델과 기준 모델 **2개**, 생성도 필요 없습니다. SFT처럼 가볍게 학습할 수 있습니다.

| | SFT | DPO |
|---|---|---|
| 데이터 | (질문, 정답) | (질문, chosen, rejected) |
| loss가 보는 것 | 정답의 확률 | chosen과 rejected의 확률 **차이** (기준 모델 대비) |
| 나쁜 답 | 배울 방법 없음 | 확률을 **내림** |

### 4-4. DPO loss를 직접 계산해 보기

먼저 "답변의 log 확률"을 재는 함수를 만듭니다. 2단계의 loss 계산과 같지만, 평균 대신 **답변 토큰들의 log 확률 합**을 구합니다.

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def response_logprob(model, question, response):
    # log π(response | prompt) = 답변 토큰들의 log 확률 합
    prompt_ids = tokenizer(build_prompt(question), add_special_tokens=False)["input_ids"]
    resp_ids = tokenizer(response, add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    ids = torch.tensor([prompt_ids + resp_ids], device=model.device)

    model.eval()
    logits = model(ids).logits[0, :-1].float()          # t번째 위치가 t+1번째 토큰을 예측
    logps = F.log_softmax(logits, dim=-1)
    targets = ids[0, 1:]
    token_logps = logps.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    return token_logps[len(prompt_ids) - 1:].sum().item()   # 답변 부분만 더함

ex = pref_raw["train"][1]
lp_w = response_logprob(sft_model, ex["prompt"], ex["chosen"])
lp_l = response_logprob(sft_model, ex["prompt"], ex["rejected"])
print(f"질문: {ex['prompt']}")
print(f"log π(chosen)   = {lp_w:8.2f}   ← {ex['chosen']}")
print(f"log π(rejected) = {lp_l:8.2f}   ← {ex['rejected']}")

학습을 시작하는 순간에는 정책 모델 = 기준 모델(SFT 모델)이므로 두 괄호 안이 모두 0이고, loss는 $-\log\sigma(0) = \log 2 \approx 0.693$입니다.
학습이 진행되면서 정책 모델의 log 확률이 바뀌면 loss가 어떻게 변하는지, 숫자를 가정해서 계산해 봅시다.

In [ ]:
def dpo_loss(pi_w, pi_l, ref_w, ref_l, beta):
    margin = beta * ((pi_w - ref_w) - (pi_l - ref_l))
    return -F.logsigmoid(torch.tensor(margin)).item(), margin

beta = config["dpo_beta"]
print("상황                                   | 암묵적 보상 차이 |  loss")
for desc, dw, dl in [
    ("학습 시작 (정책 = 기준)",                0.0,   0.0),
    ("chosen 확률 ↑ (log +2)",                2.0,   0.0),
    ("rejected 확률 ↓ (log -5)",              0.0,  -5.0),
    ("둘 다 (chosen +2, rejected -5)",        2.0,  -5.0),
    ("반대로 학습됨 (chosen -3, rejected +3)", -3.0,  3.0),
]:
    loss, margin = dpo_loss(lp_w + dw, lp_l + dl, lp_w, lp_l, beta)
    print(f"{desc:38} | {margin:+15.2f} | {loss:.3f}")

chosen을 올리거나 rejected를 내리면 loss가 줄어듭니다. 중요한 점은 **절대적인 확률이 아니라 기준 모델 대비 변화량**을 본다는 것입니다.
그래서 원래부터 확률이 낮던 긴 답(rejected)이라고 해서 저절로 유리하거나 불리하지 않습니다.

> 💡 **β의 역할:** β가 크면 작은 변화로도 loss가 빨리 줄어서, 모델이 기준에서 **조금만** 움직이고 멈춥니다(보수적). β가 작으면 loss를 줄이려고 더 많이 움직입니다. 보통 0.1 근처를 씁니다.

### 4-5. `DPOTrainer`로 학습하기

TRL의 `DPOTrainer`에 SFT 모델과 LoRA 설정을 주면,

- **정책 모델** = SFT 모델 + 새 LoRA (학습됨)
- **기준 모델** = 같은 모델에서 **LoRA만 끈 것** (= SFT 모델 그대로)

으로 처리합니다. 기준 모델을 따로 복사하지 않아도 되어서 메모리가 절약됩니다. LoRA의 숨은 장점입니다.

In [ ]:
def to_dpo_format(ex):
    return {"prompt": build_prompt(ex["prompt"]), "chosen": ex["chosen"], "rejected": ex["rejected"]}

pref_data = pref_raw.map(to_dpo_format, remove_columns=["rejected_type"])
print(pref_data["train"][0])

In [ ]:
from trl import DPOConfig, DPOTrainer

dpo_args = DPOConfig(
    output_dir=str(OUTPUT_DIR / "dpo"),
    num_train_epochs=config["dpo_epochs"],
    per_device_train_batch_size=config["dpo_batch_size"],
    per_device_eval_batch_size=config["dpo_batch_size"],
    gradient_accumulation_steps=config["dpo_grad_accum"],
    learning_rate=config["dpo_lr"],
    lr_scheduler_type="cosine",
    warmup_steps=2,
    beta=config["dpo_beta"],       # 위 수식의 β
    max_length=256,
    bf16=config["bf16"],
    fp16=config["fp16"],
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)

dpo_trainer = DPOTrainer(
    model=sft_model,
    ref_model=None,                # None + LoRA → LoRA를 끈 모델을 기준 모델로 사용
    args=dpo_args,
    train_dataset=pref_data["train"],
    eval_dataset=pref_data["eval"],
    processing_class=tokenizer,
    peft_config=make_lora_config(),
)
dpo_trainer.model.print_trainable_parameters()

In [ ]:
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
start = time.time()
dpo_trainer.train()
print(f"\nDPO 학습 시간: {(time.time() - start) / 60:.1f}분, 최대 VRAM: {peak_vram():.2f} GB")

### 4-6. DPO 학습 지표 읽기

`DPOTrainer`는 loss 말고도 DPO만의 지표를 기록합니다.

| 지표 | 뜻 | 기대하는 변화 |
|---|---|---|
| `loss` | DPO loss | 0.693에서 시작해 줄어듦 |
| `rewards/chosen` | chosen의 암묵적 보상 $\beta(\log\pi_\theta - \log\pi_{\text{ref}})$ | 0 근처 또는 ↑ |
| `rewards/rejected` | rejected의 암묵적 보상 | ↓ (음수로) |
| `rewards/margins` | chosen 보상 − rejected 보상 | ↑ |
| `rewards/accuracies` | chosen 보상 > rejected 보상인 비율 | 1.0에 가까워짐 |

In [ ]:
history = dpo_trainer.state.log_history
train_logs = [h for h in history if "loss" in h and "rewards/margins" in h]
steps = [h["step"] for h in train_logs]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(steps, [h["loss"] for h in train_logs], marker=".")
axes[0].axhline(math.log(2), color="gray", ls="--", label="log 2 (시작값)")
axes[0].set_title("DPO loss"); axes[0].legend()
axes[1].plot(steps, [h["rewards/chosen"] for h in train_logs], marker=".", label="chosen")
axes[1].plot(steps, [h["rewards/rejected"] for h in train_logs], marker=".", label="rejected")
axes[1].set_title("암묵적 보상"); axes[1].legend()
axes[2].plot(steps, [h["rewards/accuracies"] for h in train_logs], marker=".")
axes[2].set_ylim(-0.05, 1.05); axes[2].set_title("rewards/accuracies")
for ax in axes:
    ax.set_xlabel("step"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4-7. SFT 모델 vs DPO 모델

LoRA를 합친 뒤, 같은 질문으로 비교해 봅시다. 평균 길이가 줄고, 무례한 표현은 없어야 합니다.

In [ ]:
dpo_model = dpo_trainer.model.merge_and_unload()
del dpo_trainer, sft_model
free_memory()

all_answers["dpo"], all_reports["dpo"] = evaluate_model(dpo_model, "DPO")
for q, a_sft, a_dpo in zip(test_questions, all_answers["sft"], all_answers["dpo"]):
    print(f"\nQ: {q}\n[SFT] {a_sft}\n[DPO] {a_dpo}")

DPO는 **새 모범 답안 없이** "이쪽이 더 좋아"라는 비교만으로 모델의 습관을 바꿨습니다.

> ⚠️ 0.5B 모델 + 데이터 42쌍 + 몇 분 학습이라 효과가 크지 않거나 질문마다 다를 수 있습니다. 그래프의 `rewards/accuracies`가 올랐는데 답변 변화가 작다면 `dpo_epochs`나 `dpo_lr`을 조금 키워 보세요(연습 문제 2).

## 5. Stage 3: 강화학습의 핵심을 장난감으로 이해하기

DPO는 "**이미 있는** 좋은 답/나쁜 답"으로 학습했습니다(오프라인).
강화학습은 모델이 **학습 중에 직접 답을 만들고**, 채점 결과로 학습합니다(온라인). 정답을 몰라도 **채점만 할 수 있으면** 됩니다.

### GRPO (Group Relative Policy Optimization)

DeepSeek이 수학/추론 모델(DeepSeekMath, DeepSeek-R1)을 학습할 때 쓴 방법입니다. PPO에서 무거운 **가치 모델을 없앤** 버전입니다.

1. 질문 하나에 대해 답을 **G개 생성**합니다 (예: G=4)
2. 각 답을 채점합니다: $r_1, \dots, r_G$
3. **그룹 평균과 비교**해서 advantage를 구합니다: $A_i = \dfrac{r_i - \text{mean}(r)}{\text{std}(r)}$
4. 평균보다 좋은 답($A_i > 0$)은 확률을 높이고, 나쁜 답($A_i < 0$)은 낮춥니다: $\mathcal{L} = -\frac{1}{G}\sum_i A_i \log \pi_\theta(y_i \mid x)$

PPO는 "이 질문에서 기대 점수가 몇 점인가"를 가치 모델로 예측했는데, GRPO는 **같은 질문에 대한 답 여러 개의 평균**을 기대 점수로 씁니다. 모델 하나를 통째로 아낀 셈입니다.

### 장난감 예제: 답변 4개 중에서 고르는 "정책"

진짜 LLM 대신, 답변 후보 4개 중 하나를 고르는 아주 작은 정책을 순수 PyTorch로 만들어 봅니다.
정책의 파라미터는 후보별 점수(logits) 4개뿐이고, 처음에는 모두 0이라 각 답을 25%씩 고릅니다. 모델에게 정답은 알려 주지 않고, **채점 함수만** 줍니다.

In [ ]:
candidates = [
    "서울이다냥! 🐾",                                       # 짧고 냥이 말투
    "서울입니다.",                                          # 말투 없음
    "그런 것도 몰라? 서울이다.",                            # 무례함
    "서울이다냥! 그리고 또 말하자면 정말 중요한 이야기라서",  # 장황하고 끝맺지 못함
]

def reward_fn(text):
    # 사람이 정한 채점 규칙 (보상 함수)
    r = 0.0
    r += 1.0 if "냥" in text else 0.0          # 냥이 말투
    r += 1.0 if text.endswith("🐾") else 0.0    # 마무리 규칙
    r -= 2.0 if "몰라" in text else 0.0         # 무례함 벌점
    r -= len(text) / 50                        # 길수록 조금씩 감점
    return r

for c in candidates:
    print(f"보상 {reward_fn(c):+.2f}  ←  {c}")

In [ ]:
torch.manual_seed(0)
logits = torch.zeros(len(candidates), requires_grad=True)   # 정책의 파라미터
optimizer = torch.optim.Adam([logits], lr=0.05)
G = 8   # 한 번에 생성하는 답의 수 (그룹 크기)

prob_history = []
for step in range(150):
    probs = torch.softmax(logits, dim=0)
    prob_history.append(probs.detach().clone())

    # 1. 정책에서 답을 G개 샘플링 (모델이 직접 답을 만드는 단계)
    actions = torch.multinomial(probs, G, replacement=True)
    # 2. 채점
    rewards = torch.tensor([reward_fn(candidates[a]) for a in actions])
    # 3. 그룹 평균 대비 advantage
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-6)
    # 4. 좋은 답은 log 확률 ↑, 나쁜 답은 ↓
    log_probs = torch.log(probs[actions])
    loss = -(advantages * log_probs).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

prob_history = torch.stack(prob_history)
plt.figure(figsize=(8, 4))
for i, c in enumerate(candidates):
    plt.plot(prob_history[:, i], label=c[:22])
plt.xlabel("step"); plt.ylabel("선택 확률"); plt.title("GRPO 장난감: 정책이 보상 높은 답 쪽으로 이동")
plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.show()
print("최종 확률:", [f"{p:.2f}" for p in prob_history[-1].tolist()])

정답을 한 번도 알려 주지 않았는데, 정책이 **보상이 가장 높은 답** 쪽으로 옮겨 갔습니다. 이것이 강화학습의 핵심입니다.

진짜 LLM에서는

- "후보 4개 중 고르기" 대신 **토큰을 하나씩 샘플링해서 문장을 생성**하고,
- `log_probs`는 생성한 문장의 토큰 log 확률(4-4의 `response_logprob`와 같은 것)이 되고,
- 후보가 사실상 무한히 많아서, 처음에 한 번이라도 좋은 답을 만들 수 있어야 학습이 시작됩니다. **그래서 강화학습 전에 SFT를 먼저 합니다.**

> 💡 그룹 안의 답이 모두 같은 점수면 $A_i = 0$이라 아무것도 배우지 않습니다. 너무 쉬운(전부 정답) 질문이나 너무 어려운(전부 오답) 질문은 GRPO에서 학습 신호가 없습니다.

## 6. Stage 3 실습: `GRPOTrainer`로 진짜 모델 강화학습 맛보기

이제 DPO 모델에 GRPO를 해 봅니다. 사람의 선호 데이터 대신 **규칙 기반 보상 함수**를 씁니다.
이렇게 프로그램으로 확인할 수 있는 보상을 **verifiable reward**라고 하고, 수학 문제라면 "최종 답이 정답과 같은가", 코딩이라면 "테스트를 통과하는가"가 됩니다.

TRL의 보상 함수는 생성된 답변 목록 `completions`를 받아서 점수 목록을 돌려주면 됩니다.

In [ ]:
def cat_style_reward(completions, **kwargs):
    # 냥이 말투와 🐾 마무리
    return [(0.5 if "냥" in c else 0.0) + (0.5 if c.strip().endswith("🐾") else 0.0) for c in completions]

def brevity_reward(completions, **kwargs):
    # 10~60글자면 만점, 길어질수록 감점
    return [1.0 if 10 <= len(c.strip()) <= 60 else max(0.0, 1.0 - (len(c.strip()) - 60) / 60) for c in completions]

def politeness_reward(completions, **kwargs):
    return [-1.0 if any(w in c for w in RUDE_WORDS) else 0.0 for c in completions]

reward_funcs = [cat_style_reward, brevity_reward, politeness_reward]

# 시험해 보기
samples = ["하늘이 파란 건 빛이 흩어져서다냥! 🐾", "그런 것도 몰라? 알아서 찾아봐.", "하늘은 " + "아주 " * 30 + "파랗다"]
for s in samples:
    scores = [f(completions=[s])[0] for f in reward_funcs]
    print(f"{sum(scores):+.2f} {scores}  ←  {s[:40]}")

GRPO 학습용 데이터에는 **질문(prompt)만** 있으면 됩니다. 답은 모델이 만들고, 채점은 보상 함수가 합니다.

In [ ]:
grpo_data = sft_raw["train"].map(lambda ex: {"prompt": build_prompt(ex["question"])},
                                 remove_columns=sft_raw["train"].column_names)
print(grpo_data)

GRPO 설정에서 새로 나오는 값들입니다.

- **`num_generations`:** 질문 하나당 생성하는 답의 수 G. 장난감 예제의 `G`입니다.
- **`per_device_train_batch_size`:** 한 번에 처리하는 **답변** 수입니다. `num_generations`로 나누어떨어져야 합니다. (8이면 질문 2개 × 답 4개)
- **`max_completion_length`:** 생성할 답의 최대 토큰 수. 짧을수록 빠릅니다.
- **`beta`:** KL 페널티 계수. 4-2의 RLHF 수식에 있던 그 β입니다. DPO처럼 LoRA를 끈 모델이 기준 모델이 됩니다.
- **`temperature`:** 샘플링 온도. 다양한 답이 나와야 그룹 안에서 비교가 됩니다.

> ⏱️ GRPO는 매 스텝 답을 **생성**하므로 SFT/DPO보다 스텝당 훨씬 느립니다. 강화학습이 비싼 가장 큰 이유입니다.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_args = GRPOConfig(
    output_dir=str(OUTPUT_DIR / "grpo"),
    max_steps=config["grpo_steps"],
    per_device_train_batch_size=config["grpo_batch_size"],
    gradient_accumulation_steps=1,
    num_generations=config["grpo_num_generations"],
    max_completion_length=48,
    temperature=0.9,
    learning_rate=config["grpo_lr"],
    beta=config["grpo_beta"],
    bf16=config["bf16"],
    fp16=config["fp16"],
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    seed=42,
)

grpo_trainer = GRPOTrainer(
    model=dpo_model,
    reward_funcs=reward_funcs,
    args=grpo_args,
    train_dataset=grpo_data,
    processing_class=tokenizer,
    peft_config=make_lora_config(),
)

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
start = time.time()
grpo_trainer.train()
print(f"\nGRPO 학습 시간: {(time.time() - start) / 60:.1f}분, 최대 VRAM: {peak_vram():.2f} GB")

### 보상이 올라갔나?

`reward`는 세 보상 함수 점수의 합(그룹 평균)입니다. 스텝이 적어서 들쭉날쭉하지만 전체적으로 오르면 성공입니다.

In [ ]:
history = grpo_trainer.state.log_history
logs = [h for h in history if "reward" in h]
plt.figure(figsize=(7, 3.5))
plt.plot([h["step"] for h in logs], [h["reward"] for h in logs], marker=".", label="전체 보상 (평균)")
for name in ["cat_style_reward", "brevity_reward", "politeness_reward"]:
    key = f"rewards/{name}/mean" if f"rewards/{name}/mean" in logs[0] else f"rewards/{name}"
    if key in logs[0]:
        plt.plot([h["step"] for h in logs], [h[key] for h in logs], ls="--", alpha=0.7, label=name)
plt.xlabel("step"); plt.ylabel("reward"); plt.title("GRPO 보상 변화")
plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.show()

In [ ]:
grpo_model = grpo_trainer.model.merge_and_unload()
del grpo_trainer, dpo_model
free_memory()

all_answers["grpo"], all_reports["grpo"] = evaluate_model(grpo_model, "GRPO")
for q, a_dpo, a_grpo in zip(test_questions, all_answers["dpo"], all_answers["grpo"]):
    print(f"\nQ: {q}\n[DPO ] {a_dpo}\n[GRPO] {a_grpo}")

### Reward hacking 생각해 보기

우리 보상 함수는 **내용이 맞는지는 전혀 보지 않습니다.** "냥 🐾" 같은 짧고 의미 없는 답도 꽤 높은 점수를 받습니다.
학습을 오래 하면 모델이 이런 허점을 찾아낼 수 있고, 이것이 **reward hacking**입니다. (연습 문제 4에서 직접 확인해 보세요.)

실제 현장에서는 이것을 막으려고

- **KL 페널티(`beta`)** 로 기준 모델에서 너무 멀어지지 않게 하고
- 정답을 확인할 수 있는 과제(수학, 코드)를 쓰거나, 보상 함수를 여러 개 조합하고
- 학습 중에 생성된 답을 **사람이 직접 읽어 보며** 이상한 패턴이 생기는지 확인합니다.

## 7. 전체 비교

base → SFT → DPO → GRPO를 거치며 숫자가 어떻게 변했는지 한눈에 봅시다.

In [ ]:
import pandas as pd

table = pd.DataFrame(all_reports).T
table.index = ["① base", "② SFT", "③ DPO", "④ GRPO"][: len(table)]
display(table.round(2))

q = test_questions[0]
print(f"\nQ: {q}")
for name, answers in all_answers.items():
    print(f"[{name:4}] {answers[0]}")

### 저장하기 (선택)

최종 모델을 저장해 두면 2단계와 같은 방법으로 다시 불러올 수 있습니다.

In [ ]:
SAVE_DIR = OUTPUT_DIR / "final"
grpo_model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("저장 위치:", SAVE_DIR.resolve())

## 8. 정리

| 기법 | 데이터 | 모델 수 (학습 중) | 생성 필요? | 핵심 아이디어 | RTX 3060 12GB에서 0.5B |
|---|---|---|---|---|---|
| **SFT** | (질문, 정답) | 1 | ❌ | 정답의 확률 ↑ | Full도 가능 (2단계) |
| **RLHF (PPO)** | 선호 쌍 → 보상 모델 | 4 (정책, 기준, 보상, 가치) | ✅ | 보상 모델 점수 ↑ + KL 페널티 | LoRA로 겨우, 실용적이지 않음 |
| **DPO** | (질문, chosen, rejected) | 2 (LoRA면 사실상 1) | ❌ | chosen ↑, rejected ↓ (기준 대비) | 여유 있음 |
| **GRPO** | 질문 + 보상 함수 | 2 (LoRA면 사실상 1) | ✅ (G개씩) | 그룹 평균보다 좋은 답 ↑ | LoRA로 가능, 느림 |

**꼭 기억할 것**

1. **Post-training = 프리트레이닝 이후의 모든 학습.** 보통 SFT → 선호 학습(DPO/RLHF) → (필요하면) 강화학습 순서입니다.
2. **SFT는 모방**, 강화학습은 **시행착오**입니다. 수식은 둘 다 $-(\text{가중치}) \times \log \pi(\text{답})$ 꼴이고, 가중치와 답을 누가 정하느냐가 다릅니다.
3. **DPO**는 보상 모델과 PPO 없이 선호 데이터만으로 RLHF와 같은 목표를 학습합니다. 가볍고 안정적이라 요즘 오픈 모델에서 널리 쓰입니다.
4. **GRPO**는 답을 여러 개 만들어 서로 비교하는 강화학습입니다. 채점이 확실한 과제(수학, 코드)에서 추론 능력을 키우는 데 쓰입니다.
5. 기준 모델과 **KL/β**는 "너무 멀리 가지 마"라는 안전줄이고, 보상 함수의 허점을 파고드는 **reward hacking**은 강화학습의 가장 흔한 함정입니다.

### 연습 문제

1. `dpo_beta`를 0.01과 1.0으로 바꿔 DPO를 다시 해 보세요. `rewards/margins` 그래프와 답변 길이는 어떻게 달라지나요?
2. `dpo_epochs`를 늘려 보세요. 답변이 지나치게 짧아지거나 이상해지는 지점이 있나요?
3. `data/cat_preference_train.jsonl`에서 chosen과 rejected를 **서로 바꿔서** DPO를 해 보세요. 모델이 길고 장황해지나요? (선호 데이터가 모델의 성격을 정한다는 것을 확인)
4. `grpo_beta=0.0`, `grpo_steps=100`으로 바꾸고 GRPO를 오래 돌려 보세요. reward hacking(내용 없이 "냥 🐾"만 하는 등)이 생기나요?
5. 보상 함수를 직접 만들어 보세요. 예: 질문에 숫자가 있으면 답에도 숫자가 있어야 점수를 주는 `number_reward`
6. 장난감 예제(5절)의 `reward_fn`에서 길이 감점을 빼면 어떤 답이 이기나요? 보상 설계가 결과를 어떻게 바꾸는지 보세요.

### 더 알아보기

- **실제 Instruct 모델과 비교:** `MODEL_NAME`을 `"Qwen/Qwen2.5-0.5B-Instruct"`로 바꿔 base 모델 평가 셀만 돌려 보세요. Qwen 팀이 대규모 SFT + 선호 학습을 끝낸 모델입니다. 단, Instruct 모델은 `tokenizer.apply_chat_template()`로 만든 대화 형식을 써야 제대로 답합니다.
- **TRL 문서:** [SFTTrainer](https://huggingface.co/docs/trl/sft_trainer), [DPOTrainer](https://huggingface.co/docs/trl/dpo_trainer), [GRPOTrainer](https://huggingface.co/docs/trl/grpo_trainer)
- **논문:** InstructGPT(RLHF, 2022), DPO(2023), DeepSeekMath(GRPO, 2024), DeepSeek-R1(2025)